In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import re

import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
from nltk.corpus import stopwords

nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV
from sentence_transformers import SentenceTransformer

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [3]:
# Loading the dataset from google drive

file_path = '/content/drive/My Drive/legal_data/legal_cases.csv'
data = pd.read_csv(file_path)

# Exploratory Data Analysis

In [4]:
# Looking at the dimensions of data
data.shape

(24985, 4)

In [5]:
# Displaying column names
data.columns

Index(['case_id', 'case_outcome', 'case_title', 'case_text'], dtype='object')

In [6]:
# analyzing the distribution of categories in target variables
data["case_outcome"].value_counts()

,count
case_outcome,
cited,12219
referred to,4384
applied,2448
followed,2256
considered,1712
discussed,1024
distinguished,608
related,113
affirmed,113


In [7]:
# Number of classes
data["case_outcome"].nunique()

10

In [8]:
# analysing case_id
data["case_id"].nunique() # each row corresponds to one case

24985

In [9]:
# first 5 rows
data.head()

,case_id,case_outcome,case_title,case_text
0,Case1,cited,Alpine Hardwood (Aust) Pty Ltd v Hardys Pty Lt...,Ordinarily that discretion will be exercised s...
1,Case2,cited,Black v Lipovac [1998] FCA 699 ; (1998) 217 AL...,The general principles governing the exercise ...
2,Case3,cited,Colgate Palmolive Co v Cussons Pty Ltd (1993) ...,Ordinarily that discretion will be exercised s...
3,Case4,cited,Dais Studio Pty Ltd v Bullett Creative Pty Ltd...,The general principles governing the exercise ...
4,Case5,cited,Dr Martens Australia Pty Ltd v Figgins Holding...,The preceding general principles inform the ex...


In [10]:
# last 5 rows
data.tail()

,case_id,case_outcome,case_title,case_text
24980,Case25203,cited,Reches Pty Ltd v Tadiran Pty Ltd (1998) 85 FCR...,That is not confined to persons who control th...
24981,Case25204,cited,Sir Lindsay Parkinson &amp; Co Ltd v Triplan L...,Once the threshold prescribed by s 1335 is sat...
24982,Case25205,cited,Spiel v Commodity Brokers Australia Pty Ltd (I...,Once the threshold prescribed by s 1335 is sat...
24983,Case25206,distinguished,"Tullock Ltd v Walker (Unreported, Supreme Cour...",Given the extent to which Deumer stands to gai...
24984,Case25207,distinguished,Yandil Holdings Pty Ltd v Insurance Co of Nort...,"In my view, it is clear that the Court may do ..."


In [11]:
# Checking missing values - %
data.isnull().sum()/data.shape[0]

,0
case_id,0.000000
case_outcome,0.000000
case_title,0.000000
case_text,0.007044


In [12]:
# Checking missing values - number of records
data.isnull().sum()

,0
case_id,0
case_outcome,0
case_title,0
case_text,176


# Data Preparation

**Dropping records with missing case text**

In [13]:
data = data.loc[~data["case_text"].isnull()]

In [14]:
data.isnull().sum() # no more missing data

,0
case_id,0
case_outcome,0
case_title,0
case_text,0


**Dropping case_id as it is not required for model training**

In [15]:
data = data.drop(columns = ["case_id"])

**Semantic Grouping of class labels**

As there is huge class-imbalance, aggregating case outcomes into 4 groups

In [16]:
def group_outcomes(label):
    # Positive citations (case supports or relies on another)
    if label in ['cited', 'referred to', 'applied', 'followed']:
        return 'positive_citation'

    # Neutral references (case discussed or considered another)
    elif label in ['considered', 'discussed']:
        return 'neutral_citation'

    # Negative treatment (case distinguished or rejected)
    elif label in ['distinguished']:
        return 'negative_citation'

    # Approval or affirmation (case upheld or approved another)
    elif label in ['approved', 'affirmed', 'related']:
        return 'approval'

    # if unexpected label appears while testing
    else:
        return 'unknown'

In [17]:
data['case_outcome_grouped'] = data['case_outcome'].apply(group_outcomes)

In [18]:
data = data.drop(columns =["case_outcome"]) # dropping it as it is no longer required

**Separating Independent and Dependent Variables**

In [19]:
X = data.drop(columns = "case_outcome_grouped")
y = data["case_outcome_grouped"]

**train-val-test split**

In [20]:
# First splitting train-validation and test set

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# using straify = y to balance class while splitting

# Using the earlier split from train-validation , seperating them into individual components

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# final proportion train - 60%, validation - 20%,, test - 20%

print(X_train.shape)
print(y_train.shape)

print(X_val.shape)
print(y_val.shape)

print(X_test.shape)
print(y_test.shape)

(14885, 2)
(14885,)
(4962, 2)
(4962,)
(4962, 2)
(4962,)


**Label Encoding Target variable**

In [21]:
label_encoder = LabelEncoder()

In [22]:
y_train_encoded = label_encoder.fit_transform(y_train)

In [23]:
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

### Text Pre-processing

**1). Combining case title with case text**

In [24]:
X_train = X_train["case_title"]+' '+ X_train["case_text"]
X_val = X_val["case_title"]+' '+ X_val["case_text"]
X_test = X_test["case_title"]+' '+ X_test["case_text"]

**2). Basic cleaning**

In [25]:
def basic_clean(text):
    text = text.lower()    # lowercase
    text = re.sub(r'\n', ' ', text)   # remove newlines
    text = re.sub(r'[^a-z\s]', '', text)  # keep only letters
    text = re.sub(r'\s+', ' ', text).strip() # remove extra spaces
    return text

In [26]:
X_train = X_train.apply(lambda x: basic_clean(x) )
X_val = X_val.apply(lambda x: basic_clean(x) )
X_test = X_test.apply(lambda x: basic_clean(x) )

**3). Tokenization**

In [27]:
X_train = X_train.apply(lambda x: word_tokenize(x))
X_val = X_val.apply(lambda x: word_tokenize(x))
X_test = X_test.apply(lambda x: word_tokenize(x))

**4). Stop words removal**

In [28]:
stop_words = set(stopwords.words('english'))

In [29]:
legal_keep_words = {
    'shall', 'must', 'should', 'shouldn', "shouldn't",
    'can', 'couldn', "couldn't", 'may', 'mightn', "mightn't",
    'will', 'won', "won't", 'wouldn', "wouldn't",
    'not', 'no', 'nor', 'without',
    'if', 'unless', 'until', 'when', 'while', 'before', 'after',
    'here', 'there', 'where', 'within', 'between', 'under', 'upon',
    'own', 'whose'
}


In [30]:
words_to_be_removed = stop_words - legal_keep_words

In [31]:
X_train = [word for word in X_train if word not in list(words_to_be_removed)]
X_val = [word for word in X_val if word not in list(words_to_be_removed)]
X_test = [word for word in X_test if word not in list(words_to_be_removed)]

In [32]:
len(X_train), len(X_val), len(X_test)

(14885, 4962, 4962)

**5). Lemmatization**

In [33]:
lemmatizer = WordNetLemmatizer()

In [34]:
X_train1 = [
    [lemmatizer.lemmatize(word) for word in text_words]
    for text_words in X_train
]

X_val1 = [
    [lemmatizer.lemmatize(word) for word in text_words]
    for text_words in X_val
]

X_test1 = [
    [lemmatizer.lemmatize(word) for word in text_words]
    for text_words in X_test
]

**6). Remove short irrelavent words**

In [35]:
X_train1 = [word for word in X_train1 if len(word) > 2]
X_val1 = [word for word in X_val1 if len(word) > 2]
X_test1 = [word for word in X_test1 if len(word) > 2]

**7). Clean legal text**

In [36]:
def clean_legal_text(text):
    text = re.sub(r'\b(vs?|versus)\b', 'versus', text)       # normalize 'vs' to versus
    text = re.sub(r'\d+\s*u\.s\.\s*\d+', 'us_citation', text) # replace U.S. citations
    text = re.sub(r'section\s*\d+[a-z]*', 'section_ref', text)
    return text


In [37]:
X_train1 = [[clean_legal_text(word) for word in text_words] for text_words in X_train1]
X_val1 = [[clean_legal_text(word) for word in text_words] for text_words in X_val1]
X_test1 = [[clean_legal_text(word) for word in text_words] for text_words in X_test1]

**8). Joining words to sentences**

In [38]:
X_train_joined = [' '.join(tokens) for tokens in X_train1]
X_val_joined = [' '.join(tokens) for tokens in X_val1]
X_test_joined = [' '.join(tokens) for tokens in X_test1]

**9). Generating embeddings using sentence transformer**

In [39]:
model = SentenceTransformer('nlpaueb/legal-bert-base-uncased') # using legal bert

X_train_emb = model.encode(X_train_joined, show_progress_bar=True)
X_val_emb  = model.encode(X_val_joined, show_progress_bar=True)
X_test_emb  = model.encode(X_test_joined, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Batches:   0%|          | 0/466 [00:00<?, ?it/s]

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

# Model Building

Training SVM model with legal bert embeddings

In [40]:
param_grid = {'C': [0.1, 0.5, 1.0, 2.0]}
grid = GridSearchCV(
    estimator=LinearSVC(class_weight='balanced', random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1
)
grid.fit(X_train_emb, y_train)

best_svm = grid.best_estimator_

print(best_svm)

y_train_pred = best_svm.predict(X_train_emb)
y_val_pred = best_svm.predict(X_val_emb)
y_test_pred = best_svm.predict(X_test_emb)

LinearSVC(C=0.1, class_weight='balanced', random_state=42)


In [42]:
print("Train: ", f1_score(y_train, y_train_pred, average = "macro"))
print("Validation: ", f1_score(y_val, y_val_pred, average = "macro"))
print("Test: ", f1_score(y_test, y_test_pred, average = "macro"))

Train:  0.5463786710379517
Validation:  0.4285732222736873
Test:  0.41810115695262134


Saving the model to pickle file

In [43]:
import pickle

In [44]:
# with open("legal_case_classifier.pkl", "wb") as f:
#     pickle.dump(best_svm, f)